# 1주차 과제 베이스라인 — KRX 코스닥 종목 EDA

KRX OPEN API의 **코스닥 일별매매정보**를 직접 수집해 탐색합니다. 기본 기준일은 `2024-08-23`이며, 한 행은 **그 거래일의 코스닥 종목 하나**입니다. 이 자료는 여러 날짜의 종목 시계열이 아니라 하루의 시장 단면입니다.

- [코스닥 일별매매정보 API 엔드포인트](https://data-dbg.krx.co.kr/svc/apis/sto/ksq_bydd_trd)
- [KRX OPEN API 공식 이용방법](https://openapi.krx.co.kr/contents/OPP/INFO/OPPINFO003.jsp)
- 요청 파라미터: `basDd=20240823`
- 후보 고유키: `BAS_DD + ISU_CD`
- 출처 표기: 한국거래소 통계정보
- 원본 JSON과 CSV는 Git에서 제외된 로컬 `dataset/`에만 저장합니다.

이번 과제에서는 성공한 결과만큼 **시도한 코드, 만난 오류, 문제를 더 작게 나눈 과정, 다음 행동**을 중요하게 기록합니다. API 승인이 끝나지 않아도 안전하게 실행을 시도하고 상태를 설명하면 학습 기록이 됩니다.

- ✅ **기본 시도**: 수집 또는 수집 시도 후 질문 하나에 표나 그래프 하나로 답합니다.
- 🧩 **문제 분해**: 결측·무거래·이상치 후보 중 하나를 골라 확인 순서를 나눕니다.
- 🌱 **선택 탐색**: 그래프나 거래일을 추가해 질문을 넓힙니다.


## 1. 데이터 수집과 로딩

프로젝트 루트의 `.env`에 `KRX_API_KEY`를 설정하면 아래 셀에서 승인된 API를 호출할 수 있습니다. CSV가 없을 때만 API를 호출하며, 이미 수집한 파일이 있으면 캐시를 사용합니다. 인증키 노출을 막기 위해 실제 값과 요청 헤더는 출력하지 않습니다.

키가 없거나 서비스 승인이 대기 중이면 아래 셀은 오류의 종류를 보여 준 뒤 **연습용 소규모 예시 데이터**로 전환합니다. 예시 데이터는 실행 흐름을 연습하기 위한 것이며 실제 시장에 대한 결론을 내리는 데 사용하지 않습니다. 오류 메시지에는 인증키 값을 적거나 붙여 넣지 않습니다.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [start, *start.parents] if (path / '01주차' / 'krx_api.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트에서 노트북을 실행합니다.')

week_dir = PROJECT_ROOT / '01주차'
if str(week_dir) not in sys.path:
    sys.path.insert(0, str(week_dir))

BAS_DD = '20240823'
DATA_MODE = '실제 API/캐시'
collection_issue = None

try:
    from krx_api import collect_dataset, load_dataset
    report = collect_dataset('kosdaq_stocks', BAS_DD, root=PROJECT_ROOT)
    raw = load_dataset('kosdaq_stocks', BAS_DD, root=PROJECT_ROOT)
    print(f"수집 상태: {'캐시' if report['cached'] else 'API'}")
except (ModuleNotFoundError, ImportError, RuntimeError, FileNotFoundError, FileExistsError) as exc:
    DATA_MODE = '연습용 예시 데이터'
    collection_issue = f'{type(exc).__name__}: {exc}'
    print('[수집 보류]', collection_issue)
    print('인증키 값은 출력하거나 노트북에 붙여 넣지 않습니다.')
    raw = pd.DataFrame({
        'BAS_DD': pd.to_datetime(['2024-08-23'] * 6),
        'ISU_CD': pd.Series(['D00001', 'D00002', 'D00003', 'D00004', 'D00005', 'D00006'], dtype='string'),
        'ISU_NM': ['예시A', '예시B', '예시C', '예시D', '예시E', '예시F'],
        'MKT_NM': ['KOSDAQ'] * 6,
        'SECT_TP_NM': ['일반기업부', '벤처기업부', '일반기업부', '기술성장기업부', '벤처기업부', '일반기업부'],
        'TDD_CLSPRC': [1000, 990, 800, 2100, 970, 1980],
        'CMPPREVDD_PRC': [20, -10, 0, 100, -30, -20],
        'FLUC_RT': [2.04, -1.00, 0.00, 5.00, -3.00, -1.00],
        'TDD_OPNPRC': [990, 1010, 0, 2000, 1000, 2010],
        'TDD_HGPRC': [1030, 1020, 0, 2150, 1010, 2040],
        'TDD_LWPRC': [980, 980, 0, 1980, 960, 1960],
        'ACC_TRDVOL': [10000, 20000, 0, 50000, 12000, 18000],
        'ACC_TRDVAL': [10000000, 19800000, 0, 105000000, 11640000, 35640000],
        'MKTCAP': [1000000000, 1980000000, 800000000, 4200000000, 1455000000, 3960000000],
        'LIST_SHRS': [1000000, 2000000, 1000000, 2000000, 1500000, 2000000],
    })

print('데이터 모드:', DATA_MODE)
print('데이터 크기:', raw.shape)


## 2. 공식 필드명과 분석용 이름

| 공식 필드 | 분석용 이름 | 단위·의미 |
|---|---|---|
| `BAS_DD` | 기준일 | 거래일 |
| `ISU_CD`, `ISU_NM` | 종목코드, 종목명 | 종목 식별자와 표시 이름 |
| `MKT_NM`, `SECT_TP_NM` | 시장, 소속부 | 시장·소속 분류 |
| `TDD_OPNPRC`, `TDD_HGPRC`, `TDD_LWPRC`, `TDD_CLSPRC` | 시가, 고가, 저가, 종가 | 원 |
| `CMPPREVDD_PRC`, `FLUC_RT` | 대비, 등락률 | 원, % |
| `ACC_TRDVOL`, `ACC_TRDVAL` | 거래량, 거래대금 | 주, 원 |
| `MKTCAP`, `LIST_SHRS` | 시가총액, 상장주식수 | 원, 주 |


In [ ]:
column_names = {
    'BAS_DD': '기준일', 'ISU_CD': '종목코드', 'ISU_NM': '종목명',
    'MKT_NM': '시장', 'SECT_TP_NM': '소속부', 'TDD_CLSPRC': '종가',
    'CMPPREVDD_PRC': '대비', 'FLUC_RT': '등락률', 'TDD_OPNPRC': '시가',
    'TDD_HGPRC': '고가', 'TDD_LWPRC': '저가', 'ACC_TRDVOL': '거래량',
    'ACC_TRDVAL': '거래대금', 'MKTCAP': '시가총액', 'LIST_SHRS': '상장주식수',
}
df = raw.rename(columns=column_names).copy()
df.head()


## 3. 데이터 계약 검증

아래 셀은 기계적인 스키마·고유키·날짜 검증까지 수행합니다. 무거래 종목과 극단적인 등락률은 원본 정의와 거래 상태를 확인한 뒤, 과제에서 해석·처리 근거를 정리합니다.


In [ ]:
required = {
    '기준일', '종목코드', '종목명', '시장', '소속부',
    '시가', '고가', '저가', '종가', '대비', '등락률',
    '거래량', '거래대금', '시가총액', '상장주식수',
}
missing_columns = required - set(df.columns)
if missing_columns:
    raise ValueError(f'필수 열이 없습니다: {sorted(missing_columns)}')

print('기준일:', df['기준일'].dt.strftime('%Y-%m-%d').unique().tolist())
print('고유키 중복:', int(df.duplicated(['기준일', '종목코드']).sum()))
print('전체 결측 칸:', int(df.isna().sum().sum()))
print('무거래 종목:', int(df['거래량'].eq(0).sum()))
df.dtypes


## 4. 분석용 진단 열 만들기

무거래 종목은 종가는 있지만 시가·고가·저가가 0일 수 있습니다. 따라서 `거래량 > 0`인 행을 먼저 구분한 뒤 OHLC 규칙을 적용합니다. 이 열은 오류를 확정하는 대신 **검토 후보**를 표시해 원본을 확인할 수 있게 합니다.


In [ ]:
df['무거래여부'] = df['거래량'].eq(0)
df['OHLC_검토후보'] = False
traded = ~df['무거래여부']
df.loc[traded, 'OHLC_검토후보'] = (
    (df.loc[traded, '고가'] < df.loc[traded, ['시가', '종가']].max(axis=1))
    | (df.loc[traded, '저가'] > df.loc[traded, ['시가', '종가']].min(axis=1))
    | (df.loc[traded, '거래량'] < 0)
)

print('거래 발생 종목의 OHLC·거래량 검토 후보:', int(df['OHLC_검토후보'].sum()))
df[['시가', '고가', '저가', '종가', '등락률', '거래량', '거래대금', '시가총액']].describe().T


## 5. ✅ 기본 시도 — 질문 하나와 그래프 하나

먼저 **등락률은 어느 구간에 많이 모입니까?**라는 질문 하나만 확인합니다. 아래 셀은 바로 실행할 수 있는 첫 시도입니다. 실제 데이터 모드에서는 시장 단면을, 예시 데이터 모드에서는 그래프 코드의 동작만 관찰합니다.


In [ ]:
question = '등락률은 어느 구간에 많이 모입니까?'
print('질문:', question)
print('데이터 모드:', DATA_MODE)
display(df['등락률'].describe().to_frame('값'))

fig, ax = plt.subplots(figsize=(7, 4))
df['등락률'].dropna().plot(kind='hist', bins=min(30, max(5, len(df))), ax=ax)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('등락률(%)')
ax.set_title(f'코스닥 등락률 분포 — {BAS_DD}')
plt.show()


## 6. 🧩 문제 분해와 🌱 선택 탐색

🧩 **문제 분해**에서는 결측, 무거래, 이상치 후보 중 하나만 고릅니다. 작은 힌트는 `df['무거래여부'].value_counts()`처럼 상태별 개수를 먼저 세고, 해당 행 몇 개를 `head()`로 확인하는 것입니다. IQR 밖의 값을 곧바로 삭제하지 않고 원본 정의와 거래 상태를 확인합니다.

🌱 **선택 탐색**에서는 거래대금 상위 종목, 시가총액과 거래대금의 관계, 다른 거래일 중 하나를 추가합니다. 그래프를 추가할 때마다 질문 → 관찰 → 가능한 해석 → 한계를 적습니다.

## 이번 주 학습 기록

아래 네 줄을 자신의 말로 채웁니다.

- 질문 1개: `등락률은 어느 구간에 많이 모입니까?` 또는 직접 정한 질문
- 실행 또는 실행 시도 1개: 실행한 셀과 데이터 모드
- 관찰 결과 또는 오류 1개: 숫자·그래프 모양 또는 수집 보류 메시지
- 다음 행동 1개: 승인 상태 확인, 파일 확인, 집단별 비교 등

> **여기까지 하면 이번 주 기록 완료입니다.** 그래프 세 개와 다른 거래일 수집은 선택 탐색입니다.

> 인증키와 원본 데이터의 노출을 방지하기 위해 노트북 출력과 발표 화면을 확인합니다. 인증키, `.env`, 요청 헤더, 원본 JSON 대신 집계 결과와 출처만 공유합니다.
